# 11b — Measuring the machine

ch10 reported "~1.8 ms per frame" and hypothesized where it went. A
hypothesis is not an instrument. This half-step (10b, inserted at the ch10
walkthrough) builds the measurement tools the next steps will lean on —
step 11's ray-march verdict and ch15's four-target comparison must not rest
on un-decomposed wall clock.

Three tools, all satellites (the kernel was finished last chapter and was
not touched):

- **`bench.benchmark`** — BenchmarkTools-style adaptive sampling: warm up,
  TUNE evals-per-sample above the timer's resolution floor, then sample to a
  budget. The **minimum** is the estimator (noise on a quiet machine is
  strictly additive).
- **`bench.instrument`** — phase decomposition of the hot path by
  temporarily wrapping the FastRecord's seams. Zero kernel edits: `extract`
  and `launch` are plain dataclass fields.
- **`bench.gpu_timeline`** — WebGPU `timestamp-query` begin/end-of-pass
  clocks (nanoseconds by spec) splitting `launch` into encode+submit / GPU
  execution / readback. The adapter feature is requested by the demo device
  when present.

In [1]:
import pdum.dsl  # noqa: F401
from pdum.dsl import jit, viz
from pdum.dsl.bench import benchmark, instrument, phase_timeline

viz.install()


def make(cx, gain):
    @jit()
    def f(x):
        return gain * (x - cx)

    return f


f = make(0.1, 2.0)
f(1.0)
trial = benchmark(lambda: f(1.0))
print("hot path      :", trial)
rebuild = benchmark(lambda: make(0.1, 2.0)(1.0))
print("rebuild + call:", rebuild)
print()
print(f"the full factory pattern costs {(rebuild.minimum - trial.minimum) * 1e6:.2f} µs of phase A per frame")
print(f"evals/sample was TUNED to {trial.evals} — a naive one-call timing of a")
print(f"~{trial.minimum * 1e6:.0f} µs path would be mostly measuring the clock")

hot path      : Trial(min 2.46 µs · median 2.62 µs · mean 2.68 µs · 2913 samples × 32 evals)


rebuild + call: Trial(min 4.13 µs · median 4.37 µs · mean 4.53 µs · 3452 samples × 16 evals)

the full factory pattern costs 1.67 µs of phase A per frame
evals/sample was TUNED to 32 — a naive one-call timing of a
~2 µs path would be mostly measuring the clock


## Where the microseconds go

`instrument` wraps the warm record's `extract` and `launch` with timestamp
shims for a few frames, then restores them. The phases are the hit path's
anatomy from ch09, now with numbers attached — rendered by the new timeline
widget (static HTML, like every widget in this book):

In [2]:
phases = instrument(f, 1.0, frames=200)
for name in ("key+probe", "extract", "pack", "launch"):
    print(f"  {name:<10} {phases[name] * 1e6:6.2f} µs")
print(f"  {'total':<10} {phases['total'] * 1e6:6.2f} µs")
phase_timeline(phases)

  key+probe    1.05 µs
  extract      0.39 µs
  pack         0.60 µs
  launch       0.41 µs
  total        2.55 µs


Timeline(spans=[('key+probe', 0.0, 1.049586571753025e-06, 'host'), ('extract', 1.049586571753025e-06, 3.9286445826292036e-07, 'host'), ('pack', 1.4424510300159454e-06, 6.030336953699589e-07, 'host'), ('launch', 2.0454847253859045e-06, 4.1099730879068375e-07, 'host')], title='dispatch')

## The ch10 question, answered

One 256×256 GPU frame, decomposed. The `gpu` lane is the begin/end-of-pass
timestamp delta — what the silicon actually spent:

In [3]:
from pdum.dsl.bench import gpu_timeline


def gmake(cx):
    @jit(kind="simple_shader.compute")
    def k(i, j):
        return (i / 128.0 - cx) * (j / 128.0)

    return k


tl = gpu_timeline(gmake(0.5), out=(256, 256))
for name, _, dur, lane in tl.spans:
    print(f"  {name:<14} {dur * 1e6:8.1f} µs   [{lane}]")
print(f"  {'frame total':<14} {tl.total * 1e3:8.2f} ms")
tl

  encode+submit     485.9 µs   [host]
  gpu                12.2 µs   [gpu]
  readback         1426.8 µs   [host]
  frame total        1.92 ms


Timeline(spans=[('encode+submit', 0.0, 0.0004858894972130658, 'host'), ('gpu', 0.0004858894972130658, 1.2175050000000003e-05, 'gpu'), ('readback', 0.0004980645472130658, 0.0014268418308347464, 'host')], title='one GPU frame')

The verdict, with instruments instead of adjectives: the GPU computes this
kernel in single-digit **micro**seconds. The milliseconds are the blocking
readback (a full submit→wait→map round trip — the *boundary act* ch08
promised would be the only place bytes cross) plus bridge encode/submit.
A render loop that draws instead of reading back never pays the big span.

## Scaling: what actually moves

In [4]:
for side in (64, 256, 1024):
    tl = gpu_timeline(gmake(0.5), out=(side, side), frames=5)
    spans = {n: d for n, _, d, _ in tl.spans}
    print(
        f"  {side:>4}²  encode {spans['encode+submit'] * 1e6:7.1f} µs   "
        f"gpu {spans['gpu'] * 1e6:7.1f} µs   readback {spans['readback'] * 1e6:8.1f} µs"
    )
print()
print("the surprise: readback barely moves from 64² (16 KB) to 1024² (4 MB) —")
print("it is dominated by FIXED sync latency (submit→wait→map), not bandwidth.")
print("gpu time is micro-scale and quantizes noisily at one-pass resolution.")
print("host phases from the previous section never appear at this magnification.")

    64²  encode    93.5 µs   gpu     9.4 µs   readback   1374.5 µs
   256²  encode   106.6 µs   gpu    11.1 µs   readback   1383.9 µs
  1024²  encode   234.9 µs   gpu    55.3 µs   readback   1712.2 µs

the surprise: readback barely moves from 64² (16 KB) to 1024² (4 MB) —
it is dominated by FIXED sync latency (submit→wait→map), not bandwidth.
gpu time is micro-scale and quantizes noisily at one-pass resolution.
host phases from the previous section never appear at this magnification.


## Things to notice

- The thresholds from the step-9 gate table are now REAL: the suite fails if
  the warm hit path exceeds the fail line (`tests/test_bench.py`), instead
  of a notebook printing a number nobody re-reads.
- `instrument` works on ANY backend's record — the seams it wraps are the
  FastRecord contract, not a GPU detail. ch15 will point it at four targets.
- The timeline widget is ~35 lines of the viz satellite: spans in, static
  HTML out, hover for µs. No scripts, both notebook hosts.

## What we can't do yet

Per-op GPU attribution (one timestamp pair per PASS is the WebGPU deal;
finer needs vendor tools); async/double-buffered readback (the fix for the
big span is API-shaped — arrives with the graphics `draw(target)` surface);
CUDA events and Metal `gpuStartTime` (their backends, step 14). **Next:
arrays, `core.for`, and the C backend — step 11, with a ray-march verdict
these instruments can now referee.**